# Dashboard de Análisis de Calidad en Uso - MediSalud HIS
Este cuaderno de Jupyter presenta un análisis estadístico y visual del conjunto de datos `incidentes_2025.csv` conteniendo los 3,000 reportes de fallas clasificados bajo la norma **ISO/IEC 25022**.

## Librerías Requeridas
Asegúrese de ejecutar este cuaderno dentro del entorno virtual configurado donde están instaladas las librerías `pandas`, `plotly` y `openpyxl`.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Configuración de renderizado de Plotly
pio.templates.default = "plotly_white"

# Cargar el dataset clasificado
df = pd.read_csv('../data/incidentes_2025.csv')
df.head(5)

,id,fecha,modulo,descripcion,rol_usuario,sede,caracteristica_iso_25022,justificacion_tecnica
0,1210,1/2/2025,HCE,Historial de alergias no carga al abrir la fic...,Medico,Guayaquil,Libertad de Riesgo,Falla crítica que expone la seguridad del paci...
1,1509,1/2/2025,Portal Citas,El sistema no envia la confirmacion por correo...,Admision,Guayaquil,Efectividad,Falla funcional que impide completar la tarea ...
2,1786,1/2/2025,Farmacia,Duplicidad de codigos entre dos presentaciones...,Farmacia,Quito,Libertad de Riesgo,La duplicación de códigos de fármacos introduc...
3,2017,1/2/2025,Portal Citas,"Formulario confuso, abandono de registro antes...",Admision,Quito,Satisfacción,Diseño de interfaz de usuario confuso que prov...
4,2020,1/2/2025,Facturacion,El sistema no reconoce el convenio con la aseg...,Admision,Quito,Libertad de Riesgo,El no reconocimiento de convenios de asegurado...


## 1. Estadísticas Descriptivas Generales
A continuación, se presenta un resumen de la cantidad de registros por sede, rol de usuario, módulo e incidentes totales.

In [2]:
print(f"Total de Incidentes Registrados: {len(df)}")
print("\nDistribución por Sede:")
print(df['sede'].value_counts())
print("\nDistribución por Módulo (Top 5):")
print(df['modulo'].value_counts().head(5))
print("\nDistribución por Rol de Usuario:")
print(df['rol_usuario'].value_counts())

Total de Incidentes Registrados: 3000

Distribución por Sede:
sede
Quito        1035
Guayaquil     791
Cuenca        464
Ambato        386
Manta         324
Name: count, dtype: int64

Distribución por Módulo (Top 5):
modulo
HCE             769
Portal Citas    545
Facturacion     425
Telemedicina    320
App Movil       278
Name: count, dtype: int64

Distribución por Rol de Usuario:
rol_usuario
Paciente      919
Medico        689
Enfermeria    677
Admision      490
Farmacia      137
Gerencia       88
Name: count, dtype: int64


## 2. Distribución de Incidentes por Característica ISO/IEC 25022
Este gráfico de barras interactivo muestra el número total de quejas asociadas a cada una de las 5 características de Calidad en Uso de la norma ISO/IEC 25022.

In [3]:
df_caract = df['caracteristica_iso_25022'].value_counts().reset_index()
df_caract.columns = ['Característica ISO 25022', 'Cantidad']

fig_bar = px.bar(
    df_caract, 
    x='Característica ISO 25022', 
    y='Cantidad',
    color='Característica ISO 25022',
    text_auto=True,
    title='Incidentes en MediSalud por Característica ISO/IEC 25022',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_bar.update_layout(showlegend=False, xaxis_title="", yaxis_title="Número de Reportes")
fig_bar.show()

## 3. Porcentaje de Incidentes por Módulo del Sistema
Visualización del impacto relativo de los fallos en cada módulo principal de MediSalud HIS.

In [4]:
df_modulo = df['modulo'].value_counts().reset_index()
df_modulo.columns = ['Módulo', 'Cantidad']

fig_pie = px.pie(
    df_modulo, 
    names='Módulo', 
    values='Cantidad',
    title='Distribución de Fallos por Módulo del Sistema MediSalud HIS',
    color_discrete_sequence=px.colors.qualitative.Safe,
    hole=0.4
)
fig_pie.update_traces(textposition='inside', textinfo='percent+label')
fig_pie.show()

## 4. Distribución Geográfica de Calidad por Sede
Comparativa de incidentes clasificados por ciudad (sede) para identificar cuellos de botella geográficos.

In [5]:
df_sede = df.groupby(['sede', 'caracteristica_iso_25022']).size().reset_index(name='Cantidad')

fig_sede = px.bar(
    df_sede, 
    x='sede', 
    y='Cantidad', 
    color='caracteristica_iso_25022',
    title='Incidentes de Calidad en Uso por Sede Hospitalaria',
    barmode='group',
    labels={'sede': 'Sede Hospitalaria', 'Cantidad': 'Reportes', 'caracteristica_iso_25022': 'Característica ISO'},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig_sede.update_layout(xaxis_title="Ciudad / Sede", yaxis_title="Número de Reportes")
fig_sede.show()

## Conclusiones del Dashboard
1.  **Efectividad** es la categoría con más reportes (cercano al 50%), reflejando que los usuarios se enfrentan a impedimentos funcionales para terminar sus tareas clínicas o administrativas.
2.  **Libertad de Riesgo** representa un volumen muy alto (24%), lo que significa que 1 de cada 4 incidentes pone en riesgo la salud de un paciente o la estabilidad financiera del hospital. Esto exige atención correctiva inmediata.
3.  **Eficiencia** y **Satisfacción** revelan cuellos de botella en el rendimiento del servidor y frustraciones en el flujo del portal de citas, requiriendo optimización de base de datos e interfaces UI.